In [3]:
import os
import json
import requests
from dotenv import load_dotenv

In [4]:
is_localhost = True

if is_localhost:
    load_dotenv('../.env')
else:
    load_dotenv()

In [5]:
GRAFANA_URL = "http://localhost:3000"

GRAFANA_USER = os.getenv("GRAFANA_ADMIN_USER")
GRAFANA_PASSWORD = os.getenv("GRAFANA_ADMIN_PASSWORD")

PG_HOST = os.getenv("POSTGRES_HOST")
PG_DB = os.getenv("POSTGRES_DB")
PG_USER = os.getenv("POSTGRES_USER")
PG_PASSWORD = os.getenv("POSTGRES_PASSWORD")
PG_PORT = os.getenv("POSTGRES_PORT")

In [4]:
PG_HOST

'postgres'

In [6]:
import requests

# Prerequisites: Ensure variables are loaded from your environment or .env file
GRAFANA_URL = "http://localhost:3000"  # Modify to match your target environment
GRAFANA_USER = "admin"
GRAFANA_PASSWORD = "admin"

auth = (GRAFANA_USER, GRAFANA_PASSWORD)
headers = {"Content-Type": "application/json"}

def get_or_create_service_account_token():
    account_payload = {
        "name": "ProgrammaticServiceAccount",
        "role": "Admin",  # Options: Viewer, Editor, Admin
        "isDisabled": False
    }

    # Step 1: Attempt to create the Service Account
    sa_url = f"{GRAFANA_URL}/api/serviceaccounts"
    response = requests.post(sa_url, auth=auth, headers=headers, json=account_payload)

    # 201 Created (Success)
    if response.status_code == 201:
        print("Service account created successfully.")
        sa_id = response.json()["id"]
        return create_token(sa_id)

    # 400 Bad Request or 409 Conflict (Account already exists)
    elif response.status_code in [400, 409]:
        print("Service account name already exists. Fetching existing account to recreate...")
        
        # Step 2: Query existing service accounts to find the matching ID
        search_response = requests.get(f"{sa_url}/search", auth=auth)
        if search_response.status_code == 200:
            service_accounts = search_response.json().get("serviceAccounts", [])
            for sa in service_accounts:
                if sa["name"] == account_payload["name"]:
                    sa_id = sa["id"]
                    
                    # Step 3: Delete the outdated service account
                    del_response = requests.delete(f"{sa_url}/{sa_id}", auth=auth)
                    if del_response.status_code == 200:
                        print("Old service account deleted successfully.")
                        # Recurse once to cleanly spin up the fresh account
                        return get_or_create_service_account_token()
                        
        print("Failed to resolve conflicting service account.")
        return None
    else:
        print(f"Failed to communicate with Grafana API: {response.status_code} - {response.text}")
        return None

def create_token(sa_id):
    """Generates a usable bearer token linked to the specific Service Account ID."""
    token_url = f"{GRAFANA_URL}/api/serviceaccounts/{sa_id}/tokens"
    token_payload = {"name": "ProgrammaticToken"}
    
    token_response = requests.post(token_url, auth=auth, headers=headers, json=token_payload)
    if token_response.status_code == 200:
        print("Service account token created successfully.")
        return token_response.json()["key"]  # This string is your valid bearer token
    else:
        print(f"Failed to generate token: {token_response.text}")
        return None

In [ ]:
# Execute the workflow
bearer_token = get_or_create_service_account_token()
if bearer_token:
    print("\nYour usable token: Bearer {bearer_token}")

In [7]:
def create_or_update_datasource(access_token):
    PG_VERSION = os.getenv("POSTGRES_VERSION")
    PG_HOST = os.getenv("POSTGRES_HOST")
    PG_PORT = os.getenv("POSTGRES_PORT")
    PG_USER = os.getenv("POSTGRES_USER")
    PG_PASSWORD = os.getenv("POSTGRES_PASSWORD")
    PG_DB = os.getenv("POSTGRES_DB")
    
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json",
    }
    datasource_payload = {
        "name": "PostgreSQL",
        "type": "postgres",
        "url": f"{PG_HOST}:{PG_PORT}",
        "access": "proxy",
        "user": PG_USER,
        "database": PG_DB,
        "basicAuth": False,
        "isDefault": True,
        "jsonData": {"sslmode": "disable", "postgresVersion": PG_VERSION},
        "secureJsonData": {"password": PG_PASSWORD},
    }

    print("Datasource payload:")
    print(json.dumps(datasource_payload, indent=2))

    # First, try to get the existing datasource
    response = requests.get(
        f"{GRAFANA_URL}/api/datasources/name/{datasource_payload['name']}",
        headers=headers,
    )

    if response.status_code == 200:
        # Datasource exists, let's update it
        existing_datasource = response.json()
        datasource_uid = existing_datasource["uid"]

        print(f"Updating existing datasource with uid: {datasource_uid}")

        response = requests.put(
            f"{GRAFANA_URL}/api/datasources/uid/{datasource_uid}",
            headers=headers,
            json=datasource_payload,
        )
    else:
        # Datasource doesn't exist, create a new one
        print("Creating new datasource")
        response = requests.post(
            f"{GRAFANA_URL}/api/datasources", headers=headers, json=datasource_payload
        )

    print(f"Response status code: {response.status_code}")
    print(f"Response headers: {response.headers}")
    print(f"Response content: {response.text}")

    if response.status_code in [200, 201]:
        print("Datasource created or updated Successfully!!")
        return response.json().get("datasource", {}).get("uid") or response.json().get(
            "uid"
        )
    else:
        print(f"Failed to create or update datasource: {response.text}")
        return None



In [ ]:
datasource_uid = create_or_update_datasource(bearer_token)
print(datasource_uid)

In [9]:

is_localhost = True
import copy

def create_dashboard(access_token, datasource_uid):
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json",
    }

    if is_localhost:
        dashboard_file = "../app/graphana-dashboard.json"
    else:
        dashboard_file = "graphana-dashboard.json"

    try:
        with open(dashboard_file, "r") as f:
            dashboard_json = json.load(f)
            
    except FileNotFoundError:
        print(f"Error: {dashboard_file} not found.")
        return
    except json.JSONDecodeError as e:
        print(f"Error decoding {dashboard_file}: {str(e)}")
        return

    print("Dashboard JSON loaded successfully.")

    # Update datasource UID in the dashboard JSON
    panels_updated = 0
    import copy



    NEW_DATASOURCE_NAME = "PostgreSQL"
    NEW_DATASOURCE_UID = datasource_uid

    updated_dashboard = copy.deepcopy(dashboard_json)

    if "elements" in updated_dashboard["spec"]:
        for element_id, element_data in updated_dashboard["spec"]["elements"].items():
            # Drill down into spec -> data -> spec -> queries
            try:
                queries = element_data["spec"]["data"]["spec"]["queries"]
                for query in queries:
                    # Update the target datasource properties
                    if "datasource" in query["spec"]["query"]:
                        # Change the targeting reference name
                        query["spec"]["query"]["datasource"]["name"] = NEW_DATASOURCE_UID
                        query["spec"]["query"]["group"] = NEW_DATASOURCE_NAME

                        panels_updated+=1
                        print(f"{panels_updated}] Updated data source reference for {element_id}")
                        
            except KeyError:
                # Handles any panels/elements that do not have queries (text fields, rows, etc.)
                continue

    print(f"Updated datasource UID for {panels_updated} panels/targets.")
 
    # Prepare the payload
    dashboard_payload = {
        "dashboard": updated_dashboard,
        "overwrite": True
    }

    print("Sending dashboard creation request...")
    print(json.dumps(dashboard_payload, indent=2))

    response = requests.post(
        f"{GRAFANA_URL}/api/dashboards/db", headers=headers, json=dashboard_payload
    )

    print(f"Response status code: {response.status_code}")
    print(f"Response content: {response.text}")

    if response.status_code == 200:
        print("Dashboard created successfully")
        print(f"New Version: {response.json().get('version')}")
        return response.json().get("uid")
    else:
        print(f"Failed to create dashboard: {response.text}")
        return None



In [ ]:
create_dashboard(bearer_token, datasource_uid)

In [14]:
def check_grafana_status(url="http://localhost:3000"):
    try:
        # Send a GET request to the Grafana server
        response = requests.get(url)

        # Check if the response status code is 200 (OK)
        if response.status_code == 200:
            print("Grafana service is running.")
        else:
            print(f"Grafana service returned status code {response.status_code}. It may not be running correctly.")
    except requests.exceptions.RequestException as e:
        print(f"Error connecting to Grafana service: {e}")

In [13]:
check_grafana_status()

Error connecting to Grafana service: ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))
